# Pangeo-Fish による魚類の位置推定

**放流から再捕までのあいだ、魚はどこを移動していたのでしょうか？** このワークショップでは、タグが記録した水温を海洋モデルの水温や音響テレメトリーの受信記録と組み合わせ、各時刻に魚がいた可能性のある場所を推定します。

使うのは、M. Gonse ほかの研究 "Combining acoustic telemetry with archival tagging to investigate the spatial dynamic of the understudied pollack, *Pollachius pollachius*"（*Journal of Fish Biology* 106(5): 1400–1421, 2024, [doi:10.1111/jfb.15750](https://doi.org/10.1111/jfb.15750)）のタグ `A19124` です。このアーカイバルタグ（Data Storage Tag、DST）は水温と深度を記録しています。放流と再捕（または死亡）の情報も、位置を絞り込む追加の手がかりになります。

このノートブックは、すぐに使える FRA 横浜ワークショップ用の環境で動かすことを前提にしています。カーネルは **Python (pangeo-fish)** を選んでください。準備の詳細は「[事前登録](users-getting-started.md)」のページにあります。

Pangeo@EOSC の登録が承認されたら、まず [Pangeo@EOSC にログイン](https://pangeo-eosc.vm.fedcloud.eu/hub/login)してください。ログインしたらこのページに戻り、下の起動リンクを使います。ログインしていない状態では、起動リンクが正しく動かないことがあります。

**<a href="https://pangeo-eosc.vm.fedcloud.eu/hub/spawn?profile=only-choice&amp;profile-option-only-choice--image--unlisted-choice=ghcr.io%2Ftinaok%2Fpangeo-fish-notebook%3A2026.06.04-10751a5&amp;next=%2Fhub%2Fuser-redirect%2Fgit-pull%3Frepo%3Dhttps%253A%252F%252Fgithub.com%252Fpangeo-fish%252Fpangeo-fish-fra2026%26urlpath%3Dlab%252Ftree%252Fpangeo-fish-fra2026%252Fdocs%252Fpangeo-fish.ipynb%26branch%3Dmain">ここをクリックして、ワークショップの環境を起動し、このノートブックを開きます</a>**。

または、ログインしたあとで、次の URL をコピーしてブラウザのアドレスバーに貼り付けてください。

```
https://pangeo-eosc.vm.fedcloud.eu/hub/spawn?profile=only-choice&profile-option-only-choice--image--unlisted-choice=ghcr.io%2Ftinaok%2Fpangeo-fish-notebook%3A2026.06.04-10751a5&next=%2Fhub%2Fuser-redirect%2Fgit-pull%3Frepo%3Dhttps%253A%252F%252Fgithub.com%252Fpangeo-fish%252Fpangeo-fish-fra2026%26urlpath%3Dlab%252Ftree%252Fpangeo-fish-fra2026%252Fdocs%252Fpangeo-fish.ipynb%26branch%3Dmain
```

ワークショップの内容を絞るため、ここでは水温と音響テレメトリーの観測だけを使います。水深（海底地形）による尤度、潮汐、光による位置推定、行動解析は、このチュートリアルでは扱いません。海底地形や潮汐・行動解析のツールはすでにあり、光による尤度も開発済みで、ワークショップ版への組み込みを待っている段階です。これらのテーマでワークショップを希望される場合は、[Pangeo-Fish のメインリポジトリで issue を作成](https://github.com/pangeo-fish/pangeo-fish/issues/new/choose)してください。

**解析の流れ**

1. **解析の準備：** ツールを読み込み、解析の設定を選びます。
2. **水温の比較：** 候補となる場所ごとに、タグの観測値と海洋モデルの水温を比べます。
3. **等面積の格子を使う：** 候補となる場所を HEALPix のセルで表します。
4. **水温と合う場所を見つける：** 水温がよく一致する場所ほど大きな重みを与えます。
5. **音響テレメトリーの受信記録を加える：** 受信機での受信記録を使って場所を絞り込みます。
6. **手がかりを統合する：** 各時刻で、これらの観測をひとつにまとめます。
7. **移動を推定する：** 時刻と時刻のあいだに魚がどのくらいの範囲を移動できるかを推定します。
8. **位置と移動経路を推定する：** 観測と移動モデルを組み合わせます。
9. **結果を見る：** 地図を確認し、移動経路を比べ、動画を作ります。
10. **次のステップ（GFTS）：** Destination Earth で、自分の魚を将来の海水温と並べて見てみます。

観測を時間方向につなぐために、隠れマルコフモデル（Hidden Markov Model、HMM）を使います。魚の位置は直接わかりません。このモデルは、観測と移動についての仮定の両方を使って、魚がいた可能性のある場所を推定します。

## 1. 解析の準備

セルは上から順に実行してください。コードセルをクリックして **Shift+Enter** を押すと、そのセルが実行され、次のセルに移ります。計算が終わるのを待ってから次に進んでください。セルでエラーが出たら、そのあとのステップを実行する前に講師に声をかけてください。あとで問題を報告するときは、このノートブックの最後にある付録「**バグ報告用の情報（Information for bug reports）**」のセルを実行し、その出力を報告に貼り付けてください。

ワークショップの環境には、必要な Python パッケージがすでに入っています。下の準備用のセルは、変更せずにそのまま実行してください。

このセルは、ワークショップで使う計算環境を選びます。そのまま実行してください。

In [ ]:
# CPU で実行します。
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""

### 解析ツールを読み込む

In [ ]:
from pint_xarray import unit_registry as ureg
import hvplot.xarray
import xarray as xr
import sys

# sys.path.append("../")
import pangeo_fish
from pangeo_fish.xarray_update import Accessor
from pangeo_fish.helpers import multiplot_healpix, multi_map_with_synced_sliders

### 解析の設定

設定はすべてここにまとめ、使うセクションごとに分けてあります。最初は、用意された値のまま実行してください。ご自身の研究で使うときは、タグ、解析する海域、水温の許容幅、受信機の受信範囲の 4 つが特に重要な選択になります。

In [ ]:
# セクション 1 — 初期化と設定
# タグの ID。最後の代入で、このワークショップで使う A19124 を選んでいます。
tag_name = "A18828"
tag_name = "A19124"

# タグデータの場所：公開 URL、ローカルのパス、または s3:// のバケットのパス。
tag_root = "https://data-taos.ifremer.fr/demo/tag/tag/"
tag_storage_options = None
# 予備：同じタグデータを Pangeo@EOSC の MinIO にミラーしたもの（公開、読み取り専用）。
# data-taos.ifremer.fr につながらないときは、上の 2 行の代わりにこちらを使います。
# tag_root = "s3://todaka-pangeo-fish-demo/tag"
# tag_storage_options = {
#     "anon": True,
#     "client_kwargs": {"endpoint_url": "https://pangeo-eosc-minioapi.vm.fedcloud.eu"},
# }

# このワークショップで結果を保存するフォルダー。
scratch_root = "./test"

# scratch_root の下で結果を読み書きするための接続設定。
# None ではデフォルトの設定を使います。このワークショップではこれで十分です。
# scratch_root が s3:// の場合は、認証情報・プロファイルやエンドポイントの設定を指定します。
storage_options = None

# まとめて処理する時刻の数。ワークショップではこの値のままにしてください。
# 1 = チャンクごとに 1 時刻、-1 = すべての時刻を 1 つのチャンクに。
chunk_time = 1

# セクション 2 — 参照モデルの海洋データとアーカイバルタグ（DST）の観測を比べる
# Copernicus Marine の海洋モデルを選びます。有効にする代入は 1 つだけにしてください。
# どちらもワークショップの環境で動きます。最初は IBI を使ってください。

# IBI 再解析 — 領域モデル、約 3 km、日ごと（デフォルト）
# https://data.marine.copernicus.eu/product/IBI_MULTIYEAR_PHY_005_002
copernicus_model_name = "cmems_mod_ibi_phy-temp_my_0.027deg_P1D-m"

# GLORYS12 再解析 — 全球モデル、約 9 km、日ごと
# https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030
# copernicus_model_name = "cmems_mod_glo_phy_my_0.083deg_P1D-m"

# 選んだモデルを Copernicus Marine から読み込みます。このワークショップでは None のままにします。
ref_url = None

# モデルを読み込む範囲（緯度・経度）。セクション 3 と 9 でも使います。
bbox = {"latitude": [46, 51.5], "longitude": [-7, 0]}
# 魚が記録された深度に対して、モデルの海底が浅すぎるセルを除外します。
# 同じモデルの時間区間でのタグの最大深度にこの割合を掛けた値よりも
# 海底が浅いセルは除外されます。
# 範囲：0 以上。実用的な値は 0 から 1 のあいだです。
#   0    = フィルターなし：すべてのセルを残す（ワークショップの値）
#   0.8  = 海底がタグの最大深度の 80% 以上の深さであること
#          （モデルの海底地形の誤差をある程度許容する）
#   1    = 海底がタグの最大深度と同じかそれより深いこと
# 値が大きいほど多くのセルを除外します（より厳しい）。1 より大きいと、
# 海底が魚よりわずかに深いだけのセルも除外されます。
relative_depth_threshold = 0

# セクション 3 — HEALPix への再格子化
# 解析には HEALPix のセルを使います。セクション 3〜8 ではこの設定のままにします。
dims = ["cells"]

# HEALPix の解像度（おおよその相当ピクセルサイズ）
# refinement level が 1 つ上がるごとに、セルの面積は 1/4 になります。
# セルの総数 = 12 * 4**refinement_level
# 相当サイズ = 半径 6371 km の球としての地球上での sqrt(セルの面積)。
# 幾何学的な辺の長さそのものではありません。
#
# refinement_level = 10  # 約 6.4 km
refinement_level = 11    # 約 3.2 km（解像度 3 km の IBI モデルに近い）
# refinement_level = 12  # 約 1.6 km
#
# refinement level を上げると空間解像度は細かくなりますが、
# セルの数と計算コストが増えます。

# セクション 3 の HEALPix への再格子化で使う地球の形。格子と一緒に保存され、
# 地図の描画と移動の探索はそこから読み込みます。ここでは "sphere" のままにします。
ellipsoid = "sphere"
# ellipsoid = "WGS84"  # 必要な精度に対して地球の扁平率が無視できないときに推奨。

# セクション 4 — 水温にもとづく観測確率（emission probability）
# 水温の許容幅（°C）：値が大きいほど、タグとモデルの水温の差を
# より広く許容します（ガウス分布の標準偏差）。
differences_std = 0.75
# 既知の放流地点と再捕（または死亡）地点のまわりの広がり：ガウス分布の
# 標準偏差を、地球上の角度（ラジアン）で表したもの。地球の半径を掛けると
# 距離になります：1e-4 rad x 6371 km ≈ 0.64 km。
# これは HEALPix のセル 1 つ（レベル 11 で約 3.2 km）より小さいので、位置は
# 実質的に放流（または再捕）地点のセルに固定されます。
# 値が小さいほど空間的な制約が強くなります。ワークショップの値のままにしてください。
initial_std = 1e-4
recapture_std = 1e-4

# セクション 5 — 音響テレメトリーの受信による確率
# 各受信機のまわりで想定する最大受信距離。
receiver_buffer = ureg.Quantity(2000, "m")

# セクション 7 — 移動のパラメータの最適化
# 移動距離を角度に変換するための地球の半径は、HEALPix 格子に保存された
# 楕円体から求めます（"sphere" の場合は約 6371 km）。
# 移動のパラメータの探索上限を決めるための速度の目安。
# ありうる移動のひとつひとつに対する厳密な速度制限ではありません。
maximum_speed = ureg.Quantity(60, "km / day")
# 推定のときに試す sigma の値の上限に掛ける倍率。
# 1 では maximum_speed とモデルの時間間隔で決まる上限をそのまま使い、
# 2 ではその上限を 2 倍にします。推定される移動そのものを直接何倍かにするわけではありません。
# このワークショップでは 1 のままにします。動画のコマの間引きとは関係ありません。
adjustment_factor = 1
# HealpixConv で使います。LargeHealpixConv や FoscatConv では使いません。
# ワークショップではこの値のままにしてください。
truncate = 4
# パラメータ探索の数値精度。ワークショップの値のままにしてください。
tolerance = 1e-6

# セクション 8 — 状態確率と移動経路
# 移動経路の要約：各時刻の平均の位置と、もっとも確率の高いセル。
# セクション 9 の描画でも使います。
track_modes = ["mean", "mode"]
# ここで指定できるのは "speed"（km/h）と、1 つ前の点からの "distance"（km）です。
# ほかの量はあとから計算できます。セクション 8 の補足を参照してください。
additional_track_quantities = ["speed", "distance"]

# セクション 9 — 可視化と動画の作成
# 動画のフレームとフレームのあいだのデータの時刻数
# 1 = すべての時刻を含める
# 3 = 3 時刻ごとに含める
# 現在の動画作成ツールでは最後の時刻が含まれません。セクション 9 を参照してください。
time_step = 1

In [ ]:
# 解析結果を保存するルートディレクトリを決めます。
target_root = f"{scratch_root}/{tag_name}"

# 最適化で使うデフォルトのチャンクサイズを決めます。
default_chunk = {"time": chunk_time, "lat": -1, "lon": -1}
default_chunk_dims = {"time": chunk_time}
default_chunk_dims.update({d: -1 for d in dims})

### 計算の準備

このセルを実行すると Dask が起動し、使える計算資源に計算を分担させます。ここで変更する設定はありません。

In [ ]:
# 分散計算のためのローカルクラスターを準備します。
from distributed import LocalCluster

cluster = LocalCluster()
client = cluster.get_client()
client

### タグの観測データを読み込む

選んだ魚の水温・深度・イベント（放流や再捕など）の情報を読み込みます。

In [ ]:
from pangeo_fish.helpers import load_tag

tag, tag_log, time_slice = load_tag(
    tag_root=tag_root, tag_name=tag_name, storage_options=tag_storage_options
)
tag

位置を推定する前に、記録された水温と深度の時系列をプロットして、観測データを確認します。

In [ ]:
from pangeo_fish.helpers import plot_tag

plot = plot_tag(
    tag=tag,
    tag_log=tag_log,
    # 必要ならプロットをそのまま保存できます
    save_html=True,
    storage_options=storage_options,
    target_root=target_root,
)
plot

赤い点は、タグが生きた魚に付いていたとみなす期間の観測で、解析に使われます。この期間が放流と再捕（または死亡）の情報と合っているかを確認し、記録に欠けている部分がないかも見てください。

### Copernicus Marine にサインインする

次のセルを実行してください。求められたら、Copernicus Marine のユーザー名とパスワードを入力します。ログイン情報は保存され、次回以降のセッションでも使われます。すでに保存されている場合は、もう一度聞かれることはありません。

サインインできないときは、講師に声をかけるか、[Copernicus Marine の認証情報ガイド](https://help.marine.copernicus.eu/en/articles/8185007-copernicus-marine-toolbox-credentials-configuration)を参照してください。

In [ ]:
from pathlib import Path

import copernicusmarine

credentials_file = Path.home() / ".copernicusmarine" / ".copernicusmarine-credentials"

if credentials_file.is_file():
    print("Using saved Copernicus Marine credentials.")
else:
    copernicusmarine.login()  # 必要なときにユーザー名とパスワードを尋ねます。


## 2. タグと海洋モデルの水温を比べる

### 解析する海域の海水温を読み込む

ここでは [IBI 再解析](https://data.marine.copernicus.eu/product/IBI_MULTIYEAR_PHY_005_002/services)を使います。この海域をカバーする領域海洋モデルで、日ごとのデータ、解像度は約 3 km です。次のセルで、選んだ海域とタグの装着期間のデータを読み込みます。

[GLORYS12](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030/services) は全球モデルの選択肢で、日ごとのデータ、解像度は約 9 km です。モデルを切り替えるには、上の設定セルで、いま有効な `copernicus_model_name` の行の先頭に `#` を付け、もう一方の行の先頭の `#` を外します。最初は IBI で実行してください。

In [ ]:
from pangeo_fish.helpers import load_model, compute_diff

reference_model = load_model(
    uri=ref_url,
    tag_log=tag_log,
    time_slice=time_slice,
    bbox=(bbox | {"max_depth": tag_log["pressure"].max()}),
    chunk_time=chunk_time,
    remote_options={},
    copernicus_model_name=copernicus_model_name,
)

### 候補となる場所で水温を比べる

候補となる場所ごとに、タグの水温と、観測された深度でのモデルの水温を比べます。観測はモデルの時間間隔ごとにまとめられます。差が 0 に近いほど水温がよく一致していることを示しますが、似た水温の場所が複数あることもあります。

In [ ]:
%%time
diff = compute_diff(
    reference_model=reference_model,
    tag_log=tag_log,
    relative_depth_threshold=relative_depth_threshold,
    chunk_time=chunk_time,
)[0]
diff = diff.compute()
diff

### データがそろっているかを時間ごとに確認する

このプロットは、各時刻でデータのある空間セルの数を数えたものです。欠損値（Python では `NaN` と表されます）は数えません。HMM を実行する前に、この確認が大切です。

急に減っているところや、途切れているところがないかを見てください。数が多少変動するのは問題ありませんが、急に減っている場合はデータの欠損や処理の問題かもしれません。**数が 0 になっている時刻は、その時刻に使える空間データがまったくないことを意味します。先に進む前に、講師と一緒に原因を調べてください。**

水温の差が 0 というのは完全に一致しているということなので、`.count()` ではデータのあるセルとして数えられます。

In [ ]:
%%time
diff["diff"].count(["lat", "lon"]).rename("Valid grid cells").plot()

In [ ]:
%%time
diff.to_zarr(
    f"{target_root}/diff.zarr", mode="w", storage_options=storage_options, zarr_format=2
)

## 3. 候補となる場所を等面積の格子で表す

水温の比較結果を、**HEALPix** と呼ばれる格子に移します。HEALPix は等面積の離散全球格子システム（DGGS）で、どのセルも地球表面上で同じ面積を持ちます。階層構造になっているため、細かい格子と粗い格子を選べます。このワークショップでは、セルの大きさを約 **3.2 km** にしています。

このセルの大きさは、推定した魚の位置の精度ではありません。位置の不確かさは、観測、海洋モデル、移動の仮定にも左右されます。また、等面積だからといって、セルの形がすべて同じになるわけでも、空間のゆがみがすべてなくなるわけでもありません。詳しくは [HEALPix のドキュメント](https://healpix.sourceforge.io/)を参照してください。

In [ ]:
from pangeo_fish.helpers import (
    open_diff_dataset,
    regrid_dataset_hpresample,
)

In [ ]:
# 前のステップのデータセットを開きます（ノートブックをここから再開するときだけ必要です）
diff = open_diff_dataset(target_root=target_root, storage_options=storage_options)
diff

In [ ]:
reshaped, resampling_info = regrid_dataset_hpresample(
    ds=diff, refinement_level=refinement_level, ellipsoid=ellipsoid, dims=dims
)
reshaped

In [ ]:
resampling_info["bbox"] = bbox
import json

# 再格子化のメタデータと地理的な範囲を、結果と一緒に保存します。
with open(f"{target_root}/resampling_info.json", "w") as f:
    json.dump(resampling_info, f)

with open(f"{target_root}/resampling_info.json") as f:
    resampling_info = json.load(f)
resampling_info

格子を変えたので、データがそろっているかをもう一度確認します。セルの大きさや範囲が変わったため、セルの数は前と異なることがあります。急に減っているところがないかを見て、数が 0 の時刻があれば、先に進む前に原因を調べてください。

In [ ]:
reshaped["diff"].count(dims).rename("Valid HEALPix cells").plot()

In [ ]:
# 結果を保存します
reshaped.chunk(default_chunk_dims).to_zarr(
    f"{target_root}/diff-regridded.zarr",
    mode="w",
    consolidated=True,
    compute=True,
    storage_options=storage_options,
    zarr_format=2,
)

次の地図は、水温の差（`diff`）とモデルの海底の深さ（`H0`）を示しています。水温の差の地図では、0 に近い値を探してください。よく一致する場所は 1 か所に集まっていますか、それとも何か所かに分かれていますか？ このパターンが時間とともにどう変わるかも確認してください。

In [ ]:
reshaped.healpix.plot(var=["diff", "H0"], alpha=0.8)

## 4. 水温と合う場所を見つける

モデルの水温がタグの観測と一致する場所ほど、大きな重みを与えます。この重みを**観測確率（emission probability）** と呼びます（統計学では尤度（likelihood）にあたります）。観測が、それぞれの候補の場所とどのくらい整合するかを表します。

水温の差 0 を中心とするガウス分布（正規分布）の曲線を使います。設定 `differences_std` はその幅を決めます。値を大きくすると、より大きな水温の差まで許容します。放流と再捕（または死亡）の場所も、位置を絞り込む追加の手がかりになります。

`emission_pdf["pdf"]` に保存される水温の重みは、この段階ではまだ場所全体で合計しても 1 になりません。セクション 6 で手がかりを統合し、正規化します。

In [ ]:
from pangeo_fish.helpers import compute_emission_pdf

In [ ]:
# 前のステップのデータセットを開きます（ノートブックをここから再開するときだけ必要です）
differences = xr.open_dataset(
    f"{target_root}/diff-regridded.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)

# ……そして観測確率の行列を計算します
emission_pdf = compute_emission_pdf(
    diff_ds=differences,
    events_ds=tag["tagging_events"].ds,
    differences_std=differences_std,
    initial_std=initial_std,
    recapture_std=recapture_std,
    dims=dims,
    chunk_time=chunk_time,
)[0]
emission_pdf

水温の重み（`pdf`）を、放流（`initial`）と最終位置（`final`、ある場合）の地図と比べてください。水温の観測から、狭い範囲に絞り込めていますか？ それとも、可能性のある海域がいくつも残っていますか？

In [ ]:
emission_pdf.healpix.plot(["initial", "final", "pdf"])

各時刻の水温による尤度の合計 `total = sum(dims, min_count=1)` を確認します。この段階では 1 になる必要はありません。

- `NaN`：その時刻にはデータがありません。
- `0`：データはありますが、タグの水温と一致する場所がひとつもありません。

どちらかが現れたら、先に進む前に原因を調べてください。

In [ ]:
emission_pdf = emission_pdf.chunk(default_chunk_dims).persist()
total = emission_pdf["pdf"].sum(dims, min_count=1).rename("Total temperature likelihood")
print("Time steps without data (NaN):", int(total.isnull().sum()))
print("Time steps with zero total weight:", int((total == 0).sum()))
total.plot()

In [ ]:
# データセットを保存します
emission_pdf.to_zarr(
    f"{target_root}/emission_diff.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

In [ ]:
emission_pdf.attrs["sources"] = emission_pdf.attrs.get("sources", []) + [
    "temperature"
]  # これまでに加えた手がかりの種類を記録します。
emission_pdf.to_zarr(
    f"{target_root}/total_pdf.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

## 5. 音響テレメトリーの受信記録を加える

受信記録は、その時刻に魚が受信機の想定受信範囲の中にいたことを教えてくれます。この情報を使って、水温による手がかりを補いながら、魚がいた可能性のある場所を絞り込みます。

このステップには、タグのデータに音響テレメトリーの受信記録と受信機の位置が含まれている必要があります。設定 `receiver_buffer` で想定する受信範囲を決めます。この手がかりは、セクション 6 で水温の手がかりと統合します。

In [ ]:
from pangeo_fish.helpers import compute_acoustic_pdf

In [ ]:
# 前のステップの観測確率を読み込み、音響テレメトリーの受信記録にもとづく観測確率を計算します
emission_diff = xr.open_dataset(
    f"{target_root}/emission_diff.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
acoustic_pdf = compute_acoustic_pdf(
    emission_ds=emission_diff,
    tag=tag,
    receiver_buffer=receiver_buffer,
    chunk_time=chunk_time,
    dims=dims,
)[0]
acoustic_pdf = acoustic_pdf.persist()
acoustic_pdf

In [ ]:
acoustic_pdf.healpix.plot("acoustic")

次のプロットは、受信の時刻と、セル数を数えた診断用の曲線を比べるものです。受信の時刻の前後で変化があるかを見てください。

ここでは、`acoustic != 0` を満たすセルを数えています。欠損値もこの条件を満たすため、この曲線は魚がいた可能性のある場所の正確な数ではありません。この点に注意して読み取ってください。

In [ ]:
tag["acoustic"]["deployment_id"].hvplot.scatter(c="red", marker="x") * (
    acoustic_pdf["acoustic"] != 0
).sum(dim=dims).hvplot()

受信があった時刻には、想定した受信範囲のもとで、可能性のある場所が受信機の周りに一気に絞り込まれることがあります。受信記録は、位置を強く制約する手がかりです。曲線の高さを可能性のある場所の正確な数と考えるのではなく、受信の時刻と合わせて曲線の変化を読み取ってください。

次のセルでは、必要に応じて音響テレメトリーの結果を別に保存します。このあとの解析では、すでに計算した結果をそのまま使えます。

In [ ]:
acoustic_pdf.to_zarr(
    f"{target_root}/acoustic.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
)

### 手がかりをまとめる

In [ ]:
# いま保存されている手がかりを開きます。
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
current_total

In [ ]:
if "acoustic" in current_total.data_vars:
    pass
else:
    current_total = current_total.load()
    current_total.attrs["sources"] = current_total.attrs.get("sources", []) + [
        "acoustic"
    ]

    merged_pdf = current_total.merge(
        acoustic_pdf.drop_indexes(["cell_ids"]), compat="override"
    )

    merged_pdf.to_zarr(
        f"{target_root}/total_pdf.zarr",
        mode="w",
        consolidated=True,
        storage_options=storage_options,
        zarr_format=2,
    )
merged_pdf

## 6. 水温と音響テレメトリーの手がかりを統合する

水温の重みと音響テレメトリーの重みを掛け合わせ、最終位置の情報があればそれも加えます。そのうえで、各時刻で合計が 1 になるように、セル全体の重みの合計で割ります。これを**正規化**と呼びます。

この結果は、観測から見て、それぞれの候補の場所がどのくらい支持されるかを表します。場所を時間方向につなぐには、さらに移動モデルが必要です。

In [ ]:
from pangeo_fish.helpers import combine_pdfs
from pangeo_fish.helpers import normalize_pdf

In [ ]:
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
current_total

In [ ]:
emission_pdf = normalize_pdf(
    ds=current_total,
    chunks=default_chunk_dims,
    dims=dims,
    exclude=("initial", "final", "mask"),
    excluded_pdf=None,  # ("acoustic",)
)[0]

emission_pdf.to_zarr(
    f"{target_root}/combined.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)
combined = emission_pdf

emission_pdf["pdf"].sum(dims).hvplot(
    width=500, ylim=(0, 2), title="Total probability per time step"
)

確率の合計は、どの時刻でもほぼ **1** になるはずです。データのない時刻も、1 からずれた値（合計が 0）として現れます。そのような時刻があれば、移動モデルを推定する前に原因を調べてください。

In [ ]:
combined.healpix.plot("pdf")

## 7. 魚がどのくらいの範囲を移動できるかを推定する

移動モデルは、連続する時刻のあいだで、魚がいた可能性のある場所どうしをつなぎます。そのパラメータ `sigma` は、移動がどのくらい広がりうるかを決めます。値が大きいほど、より広い範囲の移動を許します。このパラメータを観測から推定します。

次のセルを実行して、移動のパラメータを推定し、保存します。探索が警告なしに終わることを確認してください。表示される `f(x)` の値は、試した `sigma` の値での当てはまりの良さを表します。探索ではいろいろな値を試すので、毎回改善しなくても問題ありません。

移動の時間間隔は、データの日付から決まります。動画の設定 `time_step` を変えても、モデルの中の魚の移動は変わりません。

<details>
<summary>補足：パラメータの探索のしくみ</summary>

探索にはブレント法（Brent's method）を使い、`1e-4` ラジアンから上限 `maximum_speed × Δt × adjustment_factor / earth_radius` までの範囲で探します。ここで Δt は観測確率の時間座標の間隔、`earth_radius` は HEALPix 格子に保存された楕円体から求めた地球の半径（`"sphere"` の場合は約 6371 km）です。速度の設定は `sigma` の探索範囲を決めるもので、ありうる移動のひとつひとつを厳密に制限するものではありません。

推定された値が探索範囲の端に近くないかを確認してください。探索が収束しても、その解がただひとつであることや、全体での最適解であることが保証されるわけではありません。付録では、別の値と比べることができます。推定したパラメータと計算の設定は、次のステップのために JSON ファイルに保存されます。

</details>

In [ ]:
from pangeo_fish.helpers import optimize_pdf

In [ ]:
# 確率分布を開きます
emission = xr.open_dataset(
    f"{target_root}/combined.zarr",
    engine="zarr",
    chunks=default_chunk_dims,
    inline_array=True,
    storage_options=storage_options,
)
emission

In [ ]:
# パラメータの範囲を決め、最適な値を探します
params = optimize_pdf(
    ds=emission,
    adjustment_factor=adjustment_factor,
    truncate=truncate,
    maximum_speed=maximum_speed,
    tolerance=tolerance,
    dims=dims,
    # 結果はそのまま保存できます
    save_parameters=True,
    storage_options=storage_options,
    target_root=f"{target_root}",
    conv_method="LargeHealpixConv",  # 高速な多重解像度の畳み込み。"HealpixConv" は時間のかかる基準となる手法
)
params

## 8. 位置を推定し、移動経路をまとめる

観測による手がかりと、推定した移動モデルを組み合わせて、各時刻にそれぞれの候補の場所に魚がいた確率を推定します。この確率分布を、次の 2 通りの方法でまとめます。

- **`mean`：** 各時刻の平均の位置。
- **`mode`：** 各時刻で確率がもっとも高いセル（最頻値）。

これらの線は推定した確率分布の要約であり、実際に観測された経路ではありません。読み取るときは、確率が地図上でどのくらい広がっているかもあわせて確認してください。移動経路の連続する点のあいだの距離を km で、それに対応する速度を km/h で計算します。

<details>
<summary>補足：ほかの移動経路の求め方と計測値</summary>

ビタビ（Viterbi）アルゴリズムは、時刻ごとに別々に要約するのではなく、もっとも確からしい位置の系列全体を求めます。このワークショップの解析では有効にしていないので、`track_modes = ["mean", "mode"]` のままにしてください。

`additional_track_quantities` の設定に指定できるのは `"speed"` と `"distance"` だけです。解析のあとで、返された移動経路から MovingPandas を使って、加速度（`add_acceleration`）、方向（`add_direction`）、方向の変化（`add_angular_difference`）、時間間隔（`add_timedelta`）を計算できます。これらは別の処理ステップです。[MovingPandas の trajectory API](https://movingpandas.readthedocs.io/en/main/api/trajectory.html) を参照してください。

</details>

In [ ]:
from pangeo_fish.helpers import predict_positions

In [ ]:
states, trajectories = predict_positions(
    target_root=f"{target_root}",
    storage_options=storage_options,
    chunks=default_chunk_dims,
    track_modes=track_modes,
    additional_track_quantities=additional_track_quantities,
    save=True,
)

`states` の確率は、各時刻で合計がほぼ **1** になるはずです。1 からずれている場合は、その時刻に欠損値や不正な値があることを意味します。

In [ ]:
states["states"].sum(dims).hvplot(
    width=500, ylim=(0, 2), title="Total probability per time step"
)

## 9. 地図と移動経路を見る

平均（mean）と最頻値（mode）の移動経路を比べ、確率の地図が時間とともにどう変わるかを見てみましょう。魚の位置がはっきり絞り込まれているのはどこでしょうか？ 可能性のある海域がいくつも残っているのはどこでしょうか？

**観測確率（emission）** の地図は、観測による手がかりを示します。**状態確率（state）** の地図は、時刻間の移動も考慮しています。両者のパターンを比べて、移動モデルが推定位置にどう影響しているかを確認してください。

### 9.1 移動経路を比べる

平均と最頻値の移動経路は、どこで一致し、どこで食い違っていますか？ 確率の地図を使って、その違いを読み解いてください。

In [ ]:
from pangeo_fish.helpers import plot_trajectories

plot_trajectories(
    target_root=f"{target_root}",
    track_modes=track_modes,
    storage_options=storage_options,
    save_html=True,
)

### 9.2 確率分布の時間変化を見る

In [ ]:
from pangeo_fish.helpers import open_distributions, render_distributions

In [ ]:
data = open_distributions(
    target_root=target_root,
    storage_options=storage_options,
    chunks=default_chunk_dims,
    chunk_time=chunk_time,
)
data

上の出力は、プロットに使えるデータの概要です。次のセルを実行すると、MP4 のアニメーションが作られ、コードのすぐ下で再生されます。

`time_step = 1` はデータのすべての時刻でフレームを作り、`3` は 3 時刻ごとにフレームを作ります。変わるのは動画のコマの間引きだけです。現在の動画作成ツールでは、この設定が 1 でも、最後の時刻は含まれません。

可能性のある場所が絞り込まれる時期や広がる時期に注目し、音響テレメトリーの受信の時刻と比べてみてください。

In [ ]:
import numpy as np

lon = data["longitude"].values  # (nx,)
lat = data["latitude"].values  # (ny,)
lon_2d, lat_2d = np.meshgrid(lon, lat)  # (ny, nx)、2 次元の地理座標

data_2d = data.rename({"latitude": "y", "longitude": "x"})
data_2d = data_2d.assign_coords(
    longitude=(("y", "x"), lon_2d),
    latitude=(("y", "x"), lat_2d),
)
data_2d

In [ ]:
video_filename = render_distributions(
    data=data_2d,
    output_path=f"{target_root}/states",
    xlim=bbox["longitude"],
    ylim=bbox["latitude"],
    time_step=time_step,
    extension="mp4",
    frames_dir="video",
    remove_frames=True,
    storage_options=storage_options,
)

In [ ]:
from IPython.display import Video, display

display(Video(filename=video_filename, embed=True))

## 10. 次のステップ：GFTS で見る、将来の海での魚の姿

結果は、次のセルで表示される出力フォルダーに保存されています（`states.zarr` と `trajectories.parq`）。これらを Destination Earth の Global Fish Tracking Service で将来の海水温と並べて表示する方法は、「[将来の海での魚の姿：GFTS](https://pangeo-fish.github.io/pangeo-fish-fra2026/gfts.html)」に進んでご覧ください。そのあと、短い[アンケート](https://pangeo-fish.github.io/pangeo-fish-fra2026/feedback.html)へのご回答をお願いします。


In [ ]:
print("Your results are in:", target_root)


## 研究で使うときは

このノートブックの手法を自分の研究で使った場合は、pangeo-fish の論文（Magin et al., 2026, *Journal of Open Source Software*, [doi:10.21105/joss.08332](https://doi.org/10.21105/joss.08332)）を引用してください。一緒に引用するデータやツール、pangeo-fish を使った研究の一覧は「[研究で pangeo-fish を使ったら：引用のしかた](cite.md)」のページにまとめています。

## 補足の付録：移動の仮定を比べる

ワークショップの本編はここまでです。移動の仮定がどう影響するかを調べたい方のために、次のセルでは、推定したモデルと、指定した 2 つの `sigma` の値によるモデルを比べます。確率の地図と移動経路がどう変わるかを見てください。

比較ごとに別の出力フォルダーを使います。最初の比較では、用意された設定のまま使ってください。より発展的な解析では、期間ごとに異なる移動のパラメータを使うこともできます。

In [ ]:
import json
import warnings
from pathlib import Path
import fsspec
from pangeo_fish.helpers import create_parameters, test_parameter
from pangeo_fish.helpers import optimize_pdf

まず、統合した観測確率の分布を別の出力フォルダーにコピーします。

In [ ]:
# 確率分布を開きます
emission = xr.open_dataset(
    f"{target_root}/combined.zarr",
    engine="zarr",
    chunks=default_chunk_dims,
    inline_array=True,
    storage_options=storage_options,
)
# 新しいフォルダーを作ります
emission.to_zarr(
    f"{target_root}/hand_sigma/combined.zarr",  # "hand_sigma" は別の名前に変えてもかまいません
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

指定した移動のパラメータを、同じフォルダーに保存します。

In [ ]:
params = create_parameters(
    sigma=[0.002708796991645453],
    sigma_indices=None,
    class_name="UpDownGaussian1DHealpix",  # 移動モデルの畳み込みクラス。
    target_root=f"{target_root}/hand_sigma",  # 上で使ったフォルダーと同じにします。
    save_parameters=True,
)

別の方法として、`test_parameter()` を使うと、統合した観測確率の分布と指定したパラメータを比較用のフォルダーに保存できます。既存のデータセットを `emission` として渡すか、`None` のままにしてセクション 6 の `combined.zarr` を読み込みます。

In [ ]:
params = test_parameter(
    emission=None,
    sigma_tested=[0.001],
    sigma_indices=None,
    Conv_method="UpDownGaussian1DHealpix",
    target_root=target_root,
    saving_root="/hand_sigma_UDG1D",
    default_chunk_dims=default_chunk_dims,
    storage_options=storage_options,
)

In [ ]:
from pangeo_fish.helpers import predict_positions

states_list = [
    predict_positions(
        target_root=f"{target_root}/{subfolder}",
        storage_options=storage_options,
        chunks=default_chunk_dims,
        track_modes=track_modes,
        additional_track_quantities=additional_track_quantities,
        save=True,
    )[0]
    for subfolder in [
        "",
        "hand_sigma",
        "hand_sigma_UDG1D",
    ]
]

同期した地図で状態確率の分布を比べ、その下で移動経路を比べます。

In [ ]:
from pangeo_fish.helpers import multi_map_with_synced_sliders, multiplot_healpix

liste = multiplot_healpix(
    [
        (states_list[0], ["states"]),
        (states_list[1], ["states"]),
        (states_list[2], ["states"]),
    ],
    refinement_level=refinement_level,
)
liste

multi_map_with_synced_sliders(liste)

In [ ]:
from pangeo_fish.helpers import plot_trajectories

traj_plots = [
    plot_trajectories(
        target_root=f"{target_root}/{subfolder}",
        track_modes=track_modes,
        storage_options=storage_options,
        save_html=True,
    ).options(title=f"Model's Folder: {subfolder}")
    for subfolder in [
        "",
        "hand_sigma",
        "hand_sigma_UDG1D",
    ]
]
(traj_plots[0] + traj_plots[1] + traj_plots[2]).cols(2)

## 付録：バグ報告用の情報（Information for bug reports）

うまく動かないときは、どのソフトウェアを使っていたかを正確に知る必要があります。次のセルを実行し（前のセルでエラーが出ていても、いつでも実行できます）、**出力をすべて**コピーして、エラーメッセージと一緒に [issue](https://github.com/pangeo-fish/pangeo-fish-fra2026/issues) やメールに貼り付けてください。

出力からは、次の 3 つのことがわかります。

- **どの計算環境か：** Jupyter サーバーの起動に使った Docker イメージ（レジストリ、名前、タグ。例：`ghcr.io/tinaok/pangeo-fish-notebook:2026.06.04-10751a5`）。タグの末尾は、イメージの作成に使った Pangeo-Fish のコミットです。
- **どの Pangeo-Fish のコードか：** そのイメージにインストールされている Pangeo-Fish のソースの Git リポジトリとコミット。
- **どのノートブックか：** このワークショップのリポジトリのコミットと、ノートブックを変更したかどうか。


In [ ]:
# バグ報告用のソフトウェア情報：出力をすべてコピーして報告に貼り付けてください。
import os
import platform
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version


def run(*cmd):
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except Exception as err:
        return f"unavailable ({type(err).__name__})"


def pkg_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"


git = shutil.which("git", path=os.path.join(sys.prefix, "bin")) or "git"
fish_src = os.environ.get("PANGEO_FISH_SRC", "/srv/pangeo-fish")

print("Docker image       :", os.environ.get("JUPYTER_IMAGE", "unknown (not set by the hub)"))
print("JupyterHub         :", os.environ.get("JUPYTERHUB_SERVICE_URL", os.environ.get("JUPYTERHUB_API_URL", "unknown")))
print("Pangeo-Fish version:", pkg_version("pangeo-fish"))
print("Pangeo-Fish source :", run(git, "-C", fish_src, "remote", "get-url", "origin"))
print("Pangeo-Fish commit :", run(git, "-C", fish_src, "log", "-1", "--format=%H (%cs) %s"))
print("Notebook repository:", run(git, "remote", "get-url", "origin"))
print("Notebook commit    :", run(git, "log", "-1", "--format=%H (%cs) %s"))
print("Notebook changes   :", run(git, "status", "--short", ".") or "none")
print("Python             :", sys.version.split()[0], "on", platform.platform())
for name in ["xarray", "dask", "numpy", "xdggs", "healpix-geo", "copernicusmarine"]:
    print(f"{name:19}:", pkg_version(name))
